<a href="https://colab.research.google.com/github/drksci/phosphene/blob/main/notebooks/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drksci/phosphene/blob/main/notebooks/01_data_prep.ipynb)

# 01 · Data prep — keyframes, dedupe, free labels

**phosphene / vtm** turns terminal byte streams into incremental A2UI. This notebook builds the corpus:

1. Stream a subset of [asciinema_terminal_recordings](https://huggingface.co/datasets/James4Ever0/asciinema_terminal_recordings) (79k casts, 18 GB uncompressed — we never extract it all).
2. Emulate each cast (pyte) → **keyframes** (screen settled ≥150 ms) → **dedupe** by layout signature across the corpus.
3. **Weak labels** for every frame from heuristic labeling functions (free) + per-cell confidence.
4. **Synthetic TUIs** with *exact* labels (free): shells, htop-likes, dialogs, editors, installers, pagers, dashboards — rendered as real ANSI through the same emulator.

Outputs `.npz` shards in `DATA/`. No API calls, CPU runtime is fine.

In [ ]:
#@title Setup: clone + install (re-run safe)
import os, sys
if not os.path.exists("/content/phosphene"):
    !git clone -q https://github.com/drksci/phosphene /content/phosphene
%cd /content/phosphene
!git pull -q
!pip install -q -e packages/vtcore -e packages/vttui -e "packages/vtm[train,data,label]"

#@markdown Persist data/models to Google Drive (survives runtime resets) or keep them in /content.
USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/phosphene"
else:
    DATA = "/content/phosphene_data"
os.makedirs(DATA, exist_ok=True)
print("DATA =", DATA)

In [ ]:
#@title Parameters
N_RECORDINGS = 4000   #@param {type:"integer"}
SPREAD = False        #@param {type:"boolean"}
PER_RECORDING = 12    #@param {type:"integer"}
N_SYNTH_TRAIN = 3000  #@param {type:"integer"}
N_SYNTH_VAL = 300     #@param {type:"integer"}
SHARD_SIZE = 4000

In [ ]:
#@title Fetch a subset of recordings (size-filtered, never the full 18 GB)
from vtm.corpus import fetch_recordings
paths = fetch_recordings("/content/recordings", n=N_RECORDINGS, spread=SPREAD)
print(len(paths), "recordings")

In [ ]:
#@title Keyframes → dedupe → weak labels → shards
import glob, time
from tqdm.auto import tqdm
from vtcore import iter_corpus
from vtm.data import build_weak_shard

for p in glob.glob(f"{DATA}/weak_*.npz"): os.remove(p)
seen, buf, n_shard, t0 = set(), [], 0, time.time()
for f in tqdm(iter_corpus(paths, per_recording=PER_RECORDING, seen=seen), desc="frames"):
    buf.append(f)
    if len(buf) == SHARD_SIZE:
        build_weak_shard(f"{DATA}/weak_{n_shard:03d}.npz", buf); buf, n_shard = [], n_shard + 1
if buf:
    build_weak_shard(f"{DATA}/weak_{n_shard:03d}.npz", buf)
print(f"{len(seen)} unique layouts in {time.time()-t0:.0f}s")

In [ ]:
#@title Synthetic TUIs with exact labels
from vtm.data import build_synth_shard
n_tr = build_synth_shard(f"{DATA}/synth_train.npz", range(N_SYNTH_TRAIN))
n_va = build_synth_shard(f"{DATA}/synth_val.npz", range(10**6, 10**6 + N_SYNTH_VAL))
print("synth frames:", n_tr, "train /", n_va, "val")

## Corpus stats

In [ ]:
import numpy as np, matplotlib.pyplot as plt, collections
from vtcore import load_frames, ROLES
from vtm.labeling.heuristics import frame_confidence

frames, roles, confs = [], [], []
for p in sorted(glob.glob(f"{DATA}/weak_*.npz")):
    fr, ex = load_frames(p); frames += fr; roles += ex["roles"]; confs += ex["conf"]
fc = np.array([frame_confidence(r, c.astype(np.float32)) for r, c in zip(roles, confs)])
sizes = collections.Counter(f.shape for f in frames).most_common(8)
hist = np.bincount(np.concatenate([r.ravel() for r in roles]), minlength=len(ROLES))

fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
ax[0].bar(ROLES[1:], hist[1:]); ax[0].tick_params(axis="x", rotation=60); ax[0].set_title("weak-label role cells")
ax[1].hist(fc, bins=40); ax[1].set_title("heuristic frame confidence (low → send to LLM)")
ax[2].barh([f"{h}x{w}" for (h, w), _ in sizes], [n for _, n in sizes]); ax[2].set_title("terminal sizes")
plt.tight_layout(); print(len(frames), "real frames")

## Look at it
Role overlay legend is shown above each gallery. Real frames carry *weak* labels; synthetic ones are exact.

In [ ]:
from vtm.viz import compare_html, show, frame_html, legend_html
import random
rng = random.Random(1)
for i in rng.sample(range(len(frames)), 4):
    show(compare_html(frames[i], {"frame": None, f"heuristics (conf {fc[i]:.2f})": roles[i]}))

In [ ]:
sfr, sex = load_frames(f"{DATA}/synth_val.npz")
show(legend_html() + "".join(frame_html(sfr[i], sex["roles"][i], title=f"synthetic #{i}", font_px=8) for i in range(0, 60, 12)))

In [ ]:
#@title Heuristic baseline vs exact synthetic labels
from vtm.labeling.heuristics import label
from vtm.metrics import confusion, report
from vtm.viz import plot_iou
rep_h = report(confusion([label(f)[0] for f in sfr], sex["roles"]))
plot_iou({"heuristics @ synth_val": rep_h}); print({k: round(v, 3) for k, v in rep_h.items() if k != "per_class"})